In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install keras

In [ ]:
try:
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf
print(tf.__version__)

In [ ]:
import string
from numpy import array
from pickle import load
from tensorflow.keras.preprocessing.text import Tokenizer
import matplotlib.pyplot as plt
import keras
import sys, time, os, warnings
warnings.filterwarnings("ignore")
import re

import numpy as np
import pandas as pd
from PIL import Image
import pickle
from collections import Counter
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.utils import plot_model
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense, BatchNormalization
from keras.layers import LSTM
from keras.layers import Embedding
from keras.layers import Dropout
from keras.layers import add
from keras.callbacks import ModelCheckpoint
from keras.preprocessing.image import load_img, img_to_array
from sklearn.utils import shuffle
from keras.applications.vgg16 import VGG16, preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

In [ ]:
def load_doc(filename):

	file = open(filename, 'r')

	text = file.read()

	file.close()
	return text


def remove_punctuation(text_original):
    text_no_punctuation = text_original.translate(string.punctuation)
    return(text_no_punctuation)



def remove_single_character(text):
    text_len_more_than1 = ""
    for word in text.split():
        if len(word) > 1:
            text_len_more_than1 += " " + word
    return(text_len_more_than1)


def remove_numeric(text,printTF=False):
    text_no_numeric = ""
    for word in text.split():
        isalpha = word.isalpha()
        if printTF:
            print("    {:10} : {:}".format(word,isalpha))
        if isalpha:
            text_no_numeric += " " + word
    return(text_no_numeric)

In [ ]:
from os import listdir

image_dir = '/content/drive/MyDrive/archive/Images'
images = [img for img in listdir(image_dir) if img.endswith(".jpg")]


descriptions_dir = '/content/drive/MyDrive/archive/captions.txt'

print("The number of jpg flies in Flicker8k: {}".format(len(images)))

In [ ]:
len(images)

In [ ]:

text = load_doc(descriptions_dir)
print(text[:330])

In [ ]:
data = pd.read_csv(descriptions_dir)

data.columns = ["filename","caption"]


data["index"] = data.groupby("filename").cumcount()


data = data[["index","filename","caption"]]

print(data.shape)

In [ ]:
def invalid_filename_check(data):
  for filenames in data["filename"]:
    found = re.search("(.(jpg)$)", filenames)
    if (found):
        pass
    else:
        print("Error file: {}".format(filenames))

In [ ]:
invalid_filename_check(data)

In [ ]:
data = data[data['filename'] != '2258277193_586949ec62.jpg.1']
data.shape

In [ ]:
def utility_counter(data):

  unique_filenames = np.unique(data.filename.values)
  print("The number of unique file names : {}".format(len(unique_filenames)))

  ct_dict = Counter(data.filename.values)
  print("We can see that all the keys are having values count = 5")
  print(ct_dict)

  print("The distribution of the number of captions for each image:")
  ct = Counter(Counter(data.filename.values).values())
  print(ct)
  return unique_filenames

In [ ]:
unique_filenames = utility_counter(data)

In [ ]:
def image_desc_plotter(data):
  npic = 5
  npix = 224
  target_size = (npix,npix,3)

  count = 1
  fig = plt.figure(figsize=(10,20))
  for jpgfnm in unique_filenames[20:25]:
      filename = image_dir + '/' + jpgfnm
      captions = list(data["caption"].loc[data["filename"]==jpgfnm].values)
      image_load = load_img(filename, target_size=target_size)

      ax = fig.add_subplot(npic,2,count,xticks=[],yticks=[])
      ax.imshow(image_load)
      count += 1

      ax = fig.add_subplot(npic,2,count)
      plt.axis('off')
      ax.plot()
      ax.set_xlim(0,1)
      ax.set_ylim(0,len(captions))
      for i, caption in enumerate(captions):
          ax.text(0,i,caption,fontsize=20)
      count += 1
  plt.show()

In [ ]:
image_desc_plotter(data)

In [ ]:
def create_vocabulary(data):
  vocab = []
  for captions in data.caption.values:
    vocab.extend(captions.split())
  print("Vocabulary Size : {}".format(len(set(vocab))))
  return vocab

In [ ]:
vocabulary = create_vocabulary(data)

In [ ]:
def df_word_count(data,vocabulary):
    ct = Counter(vocabulary)
    appen_1 = []
    appen_2 = []
    for i in ct.keys():
        appen_1.append(i)
    for j in ct.values():
        appen_2.append(j)
    data = {"word":appen_1 , "count":appen_2}
    dfword = pd.DataFrame(data)
    dfword = dfword.sort_values(by='count', ascending=False)
    dfword = dfword.reset_index()[["word","count"]]
    return(dfword)

In [ ]:
dfwordcount = df_word_count(data,vocabulary)

In [ ]:
dfwordcount.iloc[:10,:]

In [ ]:
topn = 50

def plthist(dfsub, title="The top 50 most frequently appearing words"):
    plt.figure(figsize=(30,3))
    plt.bar(dfsub.index,dfsub["count"],color ='g')
    plt.yticks(fontsize=20,color ='r')
    plt.xticks(dfsub.index,dfsub["word"],rotation=90,fontsize=20,color ='r')
    plt.title(title,fontsize=20)
    plt.show()

plthist(dfwordcount.iloc[:topn,:],
        title="The top 50 most frequently appearing words")
plthist(dfwordcount.iloc[-topn:,:],
        title="The least 50 most frequently appearing words")

In [ ]:

def text_clean(text_original):
    text = text_original.lower()
    text = remove_punctuation(text)
    # text = remove_single_character(text)
    text = remove_numeric(text)
    return(text)

for i, caption in enumerate(data.caption.values):
    newcaption = text_clean(caption)
    data["caption"].iloc[i] = newcaption

In [ ]:
clean_vocabulary = create_vocabulary(data)

In [ ]:
dfwordcount = df_word_count(data,clean_vocabulary)
plthist(dfwordcount.iloc[:topn,:],
        title="The top 50 most frequently appearing words")
plthist(dfwordcount.iloc[-topn:,:],
        title="The least 50 most frequently appearing words")

In [ ]:
def preprocess_images(data):
  all_img_name_vector = []

  for filenames in data["filename"]:
      full_image_path = image_dir+"/"+ filenames
      all_img_name_vector.append(full_image_path)
  return all_img_name_vector
all_img_name_vector = preprocess_images(data)
all_img_name_vector[:10]

In [ ]:
def preprocess_captions(data):
  total_captions = []

  for caption  in data["caption"].astype(str):
      caption = '<start> ' + caption+ ' <end>'
      total_captions.append(caption)
  return total_captions
total_captions = preprocess_captions(data)
total_captions[:10]

In [ ]:
ten_images = all_img_name_vector[:50] # Each image repeats 5 times in dataset
unique_images = np.unique(ten_images)
print(unique_images)

In [ ]:

from matplotlib.pyplot import figure, imshow, axis
from matplotlib.image import imread

def showImagesHorizontally(ten_images):
    fig = figure()
    number_of_files = len(ten_images)
    for i in range(number_of_files):
        a=fig.add_subplot(1,number_of_files,i+1)
        image = imread(ten_images[i])
        imshow(image,cmap='Greys_r', aspect='equal', interpolation='nearest')
        axis('off')

In [ ]:
showImagesHorizontally(unique_images)

In [ ]:
print("Total Images : " + str(len(all_img_name_vector)))
print("Total Captions : " + str(len(total_captions)))

In [ ]:
def data_limiter(num,total_captions,all_img_name_vector):

  train_captions, img_name_vector = shuffle(total_captions,all_img_name_vector,random_state=1)
  train_captions = train_captions[:num]
  img_name_vector = img_name_vector[:num]
  return train_captions,img_name_vector

In [ ]:
train_captions,img_name_vector = data_limiter(2000,total_captions,all_img_name_vector)

In [ ]:


print("Total Captions = {0} , Total images = {1}".format(len(train_captions),len(img_name_vector)))

In [ ]:

print("Total Captions = {0} , Total images = {1}".format(len(train_captions),len(img_name_vector)))

In [ ]:
def img_shape_finder(image):
  img= plt.imread(image)

  print("Shape of the image ==> {0} is ==> {1}".format(image.split('/')[6],img.shape))

In [ ]:
img_list=[]
for i in range(20):
  img_list.append(img_name_vector[i])

In [ ]:
for j in img_list:
  img_shape_finder(j)

In [ ]:

def image_and_shapes(image):
  img= plt.imread(image)
  plt.imshow(img)
  print("Shape of the image:{}".format(img.shape))

image_and_shapes("/content/drive/MyDrive/archive/Images/2172493537_128bc8b187.jpg")

In [ ]:
import imageio
def image_flipper(image):
  original_img = imageio.imread(image)

  plt.figure(1)



  plt.subplot(221)
  plt.imshow(original_img)



  flipped_img_tensor = tf.image.flip_left_right(original_img)
  flipped_img= flipped_img_tensor.numpy()
  plt.subplot(222)
  plt.imshow(flipped_img)



  upside_down_flip_tensor = tf.image.flip_up_down(original_img)
  upside_down_flip= upside_down_flip_tensor.numpy()
  plt.subplot(223)
  plt.imshow(upside_down_flip)



  gray_tensor = tf.image.rgb_to_grayscale(original_img)
  grayimg= gray_tensor.numpy()
  plt.subplot(224)
  plt.imshow(tf.squeeze(grayimg))



image_flipper("/content/drive/MyDrive/archive/Images/2172493537_128bc8b187.jpg")


In [ ]:
def load_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = preprocess_input(img)
    return img, image_path

img1,img1_path = load_image("/content/drive/MyDrive/archive/Images/2172493537_128bc8b187.jpg")
print("Shape after resize :", img1.shape)
plt.imshow(img1)

In [ ]:

import cv2

from matplotlib import pyplot as plt
def plot_image_histograms(image):

  img = cv2.imread(image,0)

  histr = cv2.calcHist([img],[0],None,[256],[0,256])


  fig = plt.figure(figsize=(8, 8))
  plt.subplot(2,1,1)
  plt.imshow(img)
  plt.subplot(2,1,2)
  plt.plot(histr)
  plt.show()

plot_image_histograms("/content/drive/MyDrive/archive/Images/2172493537_128bc8b187.jpg")

In [ ]:
import tensorflow as tf
modelvgg = tf.keras.applications.VGG16(include_top=True,weights=None)

In [ ]:
modelvgg.summary()

In [ ]:
image_model = tf.keras.applications.VGG16(include_top=False,weights='imagenet')
new_input = image_model.input
hidden_layer = image_model.layers[-1].output

image_features_extract_model = tf.keras.Model(new_input, hidden_layer)

In [ ]:
image_features_extract_model.summary()

In [ ]:
# Get unique images
encode_train = sorted(set(img_name_vector))
print(encode_train[:10])

In [ ]:
image_dataset = tf.data.Dataset.from_tensor_slices(encode_train)
for files in image_dataset:
    print(files.numpy())

In [ ]:
image_dataset = image_dataset.map(load_image, num_parallel_calls=tf.data.experimental.AUTOTUNE).batch(64)

In [ ]:
image_dataset

In [ ]:
from tqdm import tqdm

In [ ]:
for img, path in tqdm(image_dataset):
   batch_features = image_features_extract_model(img)
   batch_features = tf.reshape(batch_features,(batch_features.shape[0], -1, batch_features.shape[3]))
   for bf, p in zip(batch_features, path):
     path_of_feature = p.numpy().decode("utf-8")
     np.save(path_of_feature, bf.numpy())

In [ ]:
np_img =np.load('/content/drive/MyDrive/archive/Images/2172493537_128bc8b187.jpg.npy')

In [ ]:
print(np_img)
print("Shape : {}".format(np_img.shape))

In [ ]:
def tokenize_caption(top_k,train_captions):

  tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=top_k,oov_token="<unk>",filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ ')


  tokenizer.fit_on_texts(train_captions)
  train_seqs = tokenizer.texts_to_sequences(train_captions)


  tokenizer.word_index['<pad>'] = 0
  tokenizer.index_word[0] = '<pad>'



  train_seqs = tokenizer.texts_to_sequences(train_captions)
  return train_seqs, tokenizer

train_seqs , tokenizer = tokenize_caption(5000,train_captions)

In [ ]:
print(len(tokenizer.word_index))

In [ ]:

def calc_max_length(tensor):
    return max(len(t) for t in tensor)

max_length = calc_max_length(train_seqs)

In [ ]:
def calc_min_length(tensor):
    return min(len(t) for t in tensor)

min_length = calc_min_length(train_seqs)

In [ ]:
print('Max Length of any caption : Min Length of any caption = '+ str(max_length) +" : "+str(min_length))

In [ ]:
import seaborn as sns

df=pd.DataFrame()
df["sequence_length"] = data["caption"].apply(len)

sns.set()
distribution = sns.distplot(df["sequence_length"])

In [ ]:
  def padding_train_sequences(train_seqs,max_length,padding_type):
   cap_vector = tf.keras.preprocessing.sequence.pad_sequences(train_seqs, padding=padding_type,maxlen=max_length)
   return cap_vector

In [ ]:
padded_caption_vector = padding_train_sequences(train_seqs,max_length,'post')
print(padded_caption_vector.shape)

In [ ]:

padded_caption_vector

In [ ]:


img_name_train, img_name_test, caption_train, caption_test = train_test_split(img_name_vector,padded_caption_vector,test_size=0.2,random_state=0)

In [ ]:
print("Training Data : X = {0},Y = {1}".format(len(img_name_train), len(caption_train)))
print("Test Data : X = {0},Y = {1}".format(len(img_name_test), len(caption_test)))

In [ ]:
BATCH_SIZE = 64
BUFFER_SIZE = 1000

In [ ]:
def load_npy(img_name, cap):
  img_tensor = np.load(img_name.decode('utf-8')+'.npy')
  return img_tensor, cap

In [ ]:
def create_dataset(img_name_train,caption_train):


  dataset = tf.data.Dataset.from_tensor_slices((img_name_train, caption_train))


  dataset = dataset.map(lambda item1, item2: tf.numpy_function(load_npy, [item1, item2], [tf.float32, tf.int32]),num_parallel_calls=tf.data.experimental.AUTOTUNE)


  dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
  return dataset

In [ ]:


dataset = create_dataset(img_name_train,caption_train)
test_dataset = create_dataset(img_name_test,caption_test)

In [ ]:
embedding_dim = 256
units = 512
vocab_size = len(tokenizer.word_index) + 1
num_steps = len(img_name_train) // BATCH_SIZE
EPOCHS = 6
features_shape = 512
attention_features_shape = 49

In [ ]:

class VGG16_Encoder(tf.keras.Model):

    def __init__(self, embedding_dim):
        super(VGG16_Encoder, self).__init__()

        self.fc = tf.keras.layers.Dense(embedding_dim)


    def call(self, x):

        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

In [ ]:
class Rnn_Local_Decoder(tf.keras.Model):
  def __init__(self, embedding_dim, units, vocab_size):
    super(Rnn_Local_Decoder, self).__init__()
    self.units = units

    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

    self.lstm = tf.keras.layers.LSTM(self.units,kernel_initializer="glorot_uniform",return_sequences=True,return_state=True)
    self.fc1 = tf.keras.layers.Dense(self.units)

    self.dropout = tf.keras.layers.Dropout(0.5, noise_shape=None, seed=None)
    self.batchnormalization = tf.keras.layers.BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001, center=True, scale=True, beta_initializer='zeros', gamma_initializer='ones', moving_mean_initializer='zeros', moving_variance_initializer='ones', beta_regularizer=None, gamma_regularizer=None, beta_constraint=None, gamma_constraint=None)

    self.fc2 = tf.keras.layers.Dense(vocab_size)


    self.Uattn = tf.keras.layers.Dense(units)
    self.Wattn = tf.keras.layers.Dense(units)
    self.Vattn = tf.keras.layers.Dense(1)



  def call(self, x, features, hidden):



    hidden_with_time_axis = tf.expand_dims(hidden, 1)


    '''e(ij) = f(s(t-1),h(j))'''
    ''' e(ij) = Vattn(T)*tanh(Uattn * h(j) + Wattn * s(t))'''
    score = self.Vattn(tf.nn.tanh(self.Uattn(features) + self.Wattn(hidden_with_time_axis)))




    '''attention_weights(alpha(ij)) = softmax(e(ij))'''
    attention_weights = tf.nn.softmax(score, axis=1)




    ''' C(t) = Summation(j=1 to T) (attention_weights * VGG-16 features) '''
    context_vector = attention_weights * features
    context_vector = tf.reduce_sum(context_vector, axis=1)




    x = self.embedding(x)


    x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)


    output, state, c_state= self.lstm(x)

    x = self.fc1(output)


    x = tf.reshape(x, (-1, x.shape[2]))


    x= self.dropout(x)
    x= self.batchnormalization(x)

    x = self.fc2(x)

    return x, state, attention_weights

  def reset_state(self, batch_size):
    return tf.zeros((batch_size, self.units))

In [ ]:
def call(self, x, features, hidden):

    hidden_with_time_axis = tf.expand_dims(hidden, 1)
    score = self.Vattn(tf.nn.tanh(self.Uattn(features) + self.Wattn(hidden_with_time_axis)))
    attention_weights = tf.nn.softmax(score, axis=1)
    context_vector = attention_weights * features
    context_vector = tf.reduce_sum(context_vector, axis=1)
    x = self.embedding(x)
    x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

    output, state, c_state= self.lstm(x)
    x = self.fc1(output)

    x = tf.reshape(x, (-1, x.shape[2]))

    x= self.dropout(x)
    x= self.batchnormalization(x)
    x = self.fc2(x)

    return x, state, attention_weights


In [ ]:
encoder = VGG16_Encoder(embedding_dim)
decoder = Rnn_Local_Decoder(embedding_dim, units, vocab_size)

In [ ]:
def Encoder_features(img_tensor, target):
   features = encoder(img_tensor)
   return features,target,img_tensor

for (batch, (img_tensor, target)) in enumerate(dataset.take(1)):
  features,target,img_tensor= Encoder_features(img_tensor, target)

In [ ]:
target.shape

In [ ]:
img_tensor.shape

In [ ]:
features.shape

In [ ]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
  mask = tf.math.logical_not(tf.math.equal(real, 0))
  loss_ = loss_object(real, pred)

  mask = tf.cast(mask, dtype=loss_.dtype)
  loss_ *= mask


  return tf.reduce_sum(loss_) / (tf.reduce_sum(mask) + 1e-8)

In [ ]:


train_loss = tf.keras.metrics.Mean('train_loss', dtype=tf.float32)
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy('train_accuracy')

In [ ]:
import datetime
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = 'logs/gradient_tape/' + current_time + '/train'
test_log_dir = 'logs/gradient_tape/' + current_time + '/test'
train_summary_writer = tf.summary.create_file_writer(train_log_dir)
test_summary_writer = tf.summary.create_file_writer(test_log_dir)

In [ ]:
loss_plot = []

@tf.function
def train_step(img_tensor, target):
  loss = 0


  current_batch_size = tf.shape(target)[0]
  hidden = decoder.reset_state(batch_size=current_batch_size)

  dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * current_batch_size, 1)

  with tf.GradientTape() as tape:
      features = encoder(img_tensor)

      for i in range(1, target.shape[1]):

          predictions, hidden, _ = decoder(dec_input, features, hidden)

          loss += loss_function(target[:, i], predictions)


          dec_input = tf.expand_dims(target[:, i], 1)



  total_loss = loss / (int(target.shape[1]) - 1)

  trainable_variables = encoder.trainable_variables + decoder.trainable_variables

  gradients = tape.gradient(loss, trainable_variables)

  optimizer.apply_gradients(zip(gradients, trainable_variables))



  return loss, total_loss

In [ ]:
test_loss_plot = []

@tf.function
def test_step(img_tensor, target):
  loss = 0


  current_batch_size = tf.shape(target)[0]
  hidden = decoder.reset_state(batch_size=current_batch_size)

  start_token = tokenizer.word_index['<start>']
  dec_input = tf.expand_dims(tf.fill([tf.shape(target)[0]], start_token), 1)

  features = encoder(img_tensor)

  for i in range(1, target.shape[1]):

      predictions, hidden, attention_weights = decoder(dec_input, features, hidden)

      loss += loss_function(target[:, i], predictions)


      predicted_ids = tf.argmax(predictions, axis=1)
      dec_input = tf.expand_dims(predicted_ids, 1)




  total_loss = loss / (int(target.shape[1]) - 1)

  return loss, total_loss

In [ ]:
@tf.function
def train_step(img_tensor, target):

    loss = 0

    hidden = decoder.reset_state(batch_size=tf.shape(target)[0])

    start_token = tokenizer.word_index['<start>']
    dec_input = tf.expand_dims(tf.fill([tf.shape(target)[0]], start_token), 1)

    with tf.GradientTape() as tape:

        features = encoder(img_tensor)

        for i in range(1, target.shape[1]):

            predictions, hidden, _ = decoder(dec_input, features, hidden)

            loss += loss_function(target[:, i], predictions)

            dec_input = tf.expand_dims(target[:, i], 1)


    total_loss = loss / (int(target.shape[1]) - 1)

    trainable_variables = encoder.trainable_variables + decoder.trainable_variables

    gradients = tape.gradient(loss, trainable_variables)

    optimizer.apply_gradients(zip(gradients, trainable_variables))

    return loss, total_loss

In [ ]:
num_test_steps = len(img_name_test) // BATCH_SIZE

In [ ]:


for epoch in range(0,10):
    start = time.time()


    total_loss_train = 0
    for (batch, (img_tensor, target)) in enumerate(dataset):
        batch_loss, t_loss = train_step(img_tensor, target)
        total_loss_train += t_loss

    loss_plot.append(total_loss_train / num_steps)


    with train_summary_writer.as_default():
      tf.summary.scalar('LossPlotTrain', (total_loss_train/ num_steps), step=epoch)
      tf.summary.scalar('Train_loss', train_loss.result(), step=epoch)


    total_loss_test = 0
    for (batch, (img_tensor, target)) in enumerate(test_dataset):
        batch_loss, t_loss = test_step(img_tensor, target)
        total_loss_test += t_loss

    test_loss_plot.append(total_loss_test / num_test_steps)


    with test_summary_writer.as_default():
      tf.summary.scalar('LossPlotTest', (total_loss_test/ num_test_steps), step=epoch)





    print ('Epoch {} TrainLoss {:.6f} TestLoss {:.6f}'.format(epoch + 1,(total_loss_train/num_steps),(total_loss_test/num_test_steps)))
    print ('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

In [ ]:
def is_repeating(seq, max_period=6):
    n = len(seq)
    for period in range(1, max_period + 1):
        if n >= period * 2 and seq[-period:] == seq[-2*period:-period]:
            return True
    return False

In [ ]:
def evaluate(image):
    attention_plot = np.zeros((max_length, attention_features_shape))

    hidden = decoder.reset_state(batch_size=1)

    temp_input = tf.expand_dims(load_image(image)[0], 0)
    img_tensor_val = image_features_extract_model(temp_input)
    img_tensor_val = tf.reshape(img_tensor_val, (img_tensor_val.shape[0], -1, img_tensor_val.shape[3]))

    features = encoder(img_tensor_val)


    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)
    result = []

    for i in range(max_length):
        predictions, hidden, attention_weights = decoder(dec_input, features, hidden)

        attention_plot[i] = tf.reshape(attention_weights, (-1, )).numpy()



        predictions_np = predictions[0].numpy()
        for used_word in result[-3:]:
            if used_word in tokenizer.word_index:
                predictions_np[tokenizer.word_index[used_word]] = -1e9
        predicted_id = np.argmax(predictions_np)

        result.append(tokenizer.index_word[predicted_id])
        if len(result) > 6:
          break
        if is_repeating(result,max_period=6):
          break


        if tokenizer.index_word[predicted_id] == '<end>':
            return result, attention_plot

        dec_input = tf.expand_dims([predicted_id], 0)

    attention_plot = attention_plot[:len(result), :]
    return result, attention_plot

In [ ]:
import gradio as gr
from PIL import Image
import numpy as np

def caption_image(img):


    img_path = "/content/temp_image.jpg"
    img.save(img_path)


    result, _ = evaluate(img_path)


    caption = ' '.join([w for w in result if w not in ['<start>', '<end>', '<unk>']])

    return caption


interface = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Image Caption Generator"
)

interface.launch(share = True,debug = True)